In [0]:
%sql
SHOW TABLES FROM workspace.nfl;

In [0]:
%sql
USE workspace.nfl;
SELECT * FROM silver_nfl_teams;
SELECT * FROM silver_team_season_records 
ORDER BY win_percentage DESC
LIMIT 5;



In [0]:
%sql
USE workspace.nfl;

-- Create or update the final gold analytics table in the catalog
-- CREATE OR REPLACE TABLE workspace.nfl.gold_strength_of_schedule AS

-- create a CTE that gets the schedule of each team and all their opponents
WITH opponent_matchups AS (
    SELECT
        CAST(regexp_extract(_metadata.file_name, 'nfl_schedule_(\\d{4})', 1) AS INT) AS season_year, -- gets year by looking for 4 digits in a row
        team.id as team_id,
        team.team.name as team_name, -- Pulls the string name straight from the JSON object
        FILTER(comp.competitors, x -> x.id != team.id)[0].id AS opponent_id -- function check team id and matches it with opponent id 
    FROM read_files('/Volumes/workspace/nfl/bronze_landing/nfl_schedule_*.json', format => 'json') -- read from each file in landing zone
    LATERAL VIEW explode(events) AS event
    LATERAL VIEW explode(event.competitions) AS comp
    LATERAL VIEW explode(comp.competitors) AS team
)
SELECT 
    m.season_year,
    m.team_name,
    ROUND(AVG(r.win_percentage), 3) AS strength_of_schedule -- join opponent match up and schedul to get the opponent win percentage to see how good the team i
FROM opponent_matchups m
JOIN silver_team_season_records r 
    ON m.opponent_id = r.team_id 
    AND m.season_year = r.season_year
GROUP BY m.season_year, m.team_name
ORDER BY m.season_year DESC, strength_of_schedule DESC;